In [1]:
import kagglehub
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

In [2]:
#download dataset from Kaggle
path = kagglehub.dataset_download('new-york-city/nyc-inspections')

csv_file = next(os.path.join(path, f) for f in os.listdir(path) if f.endswith('.csv'))

df_raw = pd.read_csv(csv_file, dtype={"PHONE": "string"}, encoding='latin-1', low_memory=False)
df = df_raw.copy()
df['CUISINE DESCRIPTION'] = df['CUISINE DESCRIPTION'].str.replace(
    r'Caf[^\s/]+', 'Café', regex=True
)

print(df.shape)

100%|██████████| 21.5M/21.5M [00:00<00:00, 90.2MB/s]

Extracting files...


(399918, 18)


In [3]:
# Fix data types
for col in ['INSPECTION DATE', 'GRADE DATE', 'RECORD DATE']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

df['ZIPCODE'] = df['ZIPCODE'].apply(lambda x: str(int(x)).zfill(5) if pd.notna(x) else np.nan)

df['CAMIS'] = df['CAMIS'].astype(str)

In [4]:
# drop exact duplicate rows
df = df.drop_duplicates(keep='first')
print(f'Rows after dedup: {len(df):,}')

Rows after dedup: 399,907


In [5]:
#BORO has 9 rows with the word missing instead of NaN
df['BORO'] = df['BORO'].replace('Missing', np.nan)

In [6]:
# 1135 rows have inspection date = 1900-01-01
sentinel_mask = df['INSPECTION DATE'] == pd.Timestamp('1900-01-01')
df['IS_SENTINEL_DATE'] = sentinel_mask
df.loc[sentinel_mask, 'INSPECTION DATE'] = pd.NaT

In [7]:
#negative scores are impossible in the NYC grading system
neg_mask = df['SCORE'] < 0
df['SCORE_INVALID'] = neg_mask
df.loc[neg_mask, 'SCORE'] = np.nan

# scores > 100 are extreme but possible
df['SCORE_EXTREME'] = df['SCORE'] > 100

In [8]:
df['HAS_GRADE'] = df['GRADE'].notna()

df['PHONE'] = df['PHONE'].astype("string").str.replace(r'\.0$', '', regex=True).str.replace(r'\D', '', regex=True)

df['PHONE'] = df['PHONE'].where(df['PHONE'].str.len() == 10, None)

In [9]:
df['VIOLATION_MISMATCH'] = df['VIOLATION CODE'].isna() ^ df['VIOLATION DESCRIPTION'].isna()

In [10]:
df = df.drop(columns=['RECORD DATE'])

In [11]:
#drop rows with no real inspection date
df = df[df['IS_SENTINEL_DATE'] == False]

#drop rows where violation code and description don't match
df = df[df['VIOLATION_MISMATCH'] == False]

In [12]:
# Final checks
print(f'Shape:{df.shape}')
print(f'udplicates:{df.duplicated().sum()}')
print(f'negative scores:{(df["SCORE"] < 0).sum()}')
print(f'dentinel dates:{(df["INSPECTION DATE"] == pd.Timestamp("1900-01-01")).sum()}')
print(f'BORO Missing left:{(df["BORO"] == "Missing").sum()}')

Shape:(398302, 22)
udplicates:0
negative scores:0
dentinel dates:0
BORO Missing left:0


In [13]:
df.to_csv('nyc_restaurant_inspections_CLEAN.csv', index=False, encoding='utf-8')

In [14]:
print(df.shape)
print(df.dtypes)
print(df.head(3))

(398302, 22)
CAMIS                            object
DBA                              object
BORO                             object
BUILDING                         object
STREET                           object
ZIPCODE                          object
PHONE                    string[python]
CUISINE DESCRIPTION              object
INSPECTION DATE          datetime64[ns]
ACTION                           object
VIOLATION CODE                   object
VIOLATION DESCRIPTION            object
CRITICAL FLAG                    object
SCORE                           float64
GRADE                            object
GRADE DATE               datetime64[ns]
INSPECTION TYPE                  object
IS_SENTINEL_DATE                   bool
SCORE_INVALID                      bool
SCORE_EXTREME                      bool
HAS_GRADE                          bool
VIOLATION_MISMATCH                 bool
dtype: object
      CAMIS                DBA       BORO BUILDING         STREET ZIPCODE  \
0  40511702  NOT

In [15]:
print("=== CRITICAL FLAG ===")
print(df['CRITICAL FLAG'].value_counts(dropna=False))

print("\n=== GRADE ===")
print(df['GRADE'].value_counts(dropna=False))

print("\n=== SCORE stats ===")
print(df['SCORE'].describe())

=== CRITICAL FLAG ===
CRITICAL FLAG
Critical          220080
Not Critical      172853
Not Applicable      5369
Name: count, dtype: int64

=== GRADE ===
GRADE
NaN               203092
A                 154029
B                  28144
C                   6985
Not Yet Graded      2592
Z                   2101
P                   1359
Name: count, dtype: int64

=== SCORE stats ===
count    376240.000000
mean         18.917651
std          12.956042
min           0.000000
25%          11.000000
50%          15.000000
75%          24.000000
max         151.000000
Name: SCORE, dtype: float64


In [16]:
model_df = df[df['CRITICAL FLAG'] != 'Not Applicable'].copy()

model_df['TARGET'] = (model_df['CRITICAL FLAG'] == 'Critical').astype(int)

features = ['BORO', 'CUISINE DESCRIPTION', 'INSPECTION TYPE', 'SCORE']
model_df = model_df[features + ['TARGET']].dropna()

print(model_df.shape)
print(model_df['TARGET'].value_counts())

(374659, 5)
TARGET
1    219994
0    154665
Name: count, dtype: int64


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

cat_cols = ['BORO', 'CUISINE DESCRIPTION', 'INSPECTION TYPE']
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col])
    encoders[col] = le

X = model_df[features]
y = model_df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (299727, 4), Test: (74932, 4)


In [18]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print("Training done")

Training done


In [19]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Not Critical', 'Critical']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}')

              precision    recall  f1-score   support

Not Critical       0.64      0.25      0.36     31001
    Critical       0.63      0.90      0.74     43931

    accuracy                           0.63     74932
   macro avg       0.64      0.58      0.55     74932
weighted avg       0.63      0.63      0.58     74932

ROC-AUC: 0.6258


In [20]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

SCORE                  0.934945
CUISINE DESCRIPTION    0.027945
INSPECTION TYPE        0.025460
BORO                   0.011649
dtype: float64


In [21]:
features_no_score = ['BORO', 'CUISINE DESCRIPTION', 'INSPECTION TYPE']

X_train2 = X_train[features_no_score]
X_test2 = X_test[features_no_score]

model2 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model2.fit(X_train2, y_train)

y_pred2 = model2.predict(X_test2)
y_proba2 = model2.predict_proba(X_test2)[:, 1]

print(classification_report(y_test, y_pred2, target_names=['Not Critical', 'Critical']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba2):.4f}')

              precision    recall  f1-score   support

Not Critical       0.59      0.03      0.06     31001
    Critical       0.59      0.98      0.74     43931

    accuracy                           0.59     74932
   macro avg       0.59      0.51      0.40     74932
weighted avg       0.59      0.59      0.46     74932

ROC-AUC: 0.5348


In [22]:
insp = df[df['GRADE'].isin(['A', 'B', 'C'])].copy()

agg = insp.groupby(['CAMIS', 'INSPECTION DATE']).agg(
    BORO=('BORO', 'first'),
    CUISINE=('CUISINE DESCRIPTION', 'first'),
    SCORE=('SCORE', 'first'),
    TOTAL_VIOLATIONS=('VIOLATION CODE', 'count'),
    CRITICAL_VIOLATIONS=('CRITICAL FLAG', lambda x: (x == 'Critical').sum()),
    GRADE=('GRADE', 'first')
).reset_index()

agg['TARGET'] = (agg['GRADE'] == 'A').astype(int)
agg = agg.dropna(subset=['SCORE', 'BORO', 'CUISINE'])

print(agg.shape)
print(agg['TARGET'].value_counts())

(82733, 9)
TARGET
1    73430
0     9303
Name: count, dtype: int64


In [23]:
features2 = ['BORO', 'CUISINE', 'SCORE', 'TOTAL_VIOLATIONS', 'CRITICAL_VIOLATIONS']

for col in ['BORO', 'CUISINE']:
    le = LabelEncoder()
    agg[col] = le.fit_transform(agg[col].astype(str))

X2 = agg[features2]
y2 = agg['TARGET']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)

print(f'Train: {X2_train.shape}, Test: {X2_test.shape}')
print(f'Test class balance:\n{y2_test.value_counts()}')

Train: (66186, 5), Test: (16547, 5)
Test class balance:
TARGET
1    14686
0     1861
Name: count, dtype: int64


In [24]:
model3 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42,
                                 n_jobs=-1, class_weight='balanced')
model3.fit(X2_train, y2_train)

y_pred3 = model3.predict(X2_test)
y_proba3 = model3.predict_proba(X2_test)[:, 1]

print(classification_report(y2_test, y_pred3, target_names=['Not A', 'A']))
print(f'ROC-AUC: {roc_auc_score(y2_test, y_proba3):.4f}')

              precision    recall  f1-score   support

       Not A       0.99      0.96      0.98      1861
           A       1.00      1.00      1.00     14686

    accuracy                           1.00     16547
   macro avg       0.99      0.98      0.99     16547
weighted avg       1.00      1.00      1.00     16547

ROC-AUC: 0.9926


In [25]:
importances2 = pd.Series(model3.feature_importances_, index=features2).sort_values(ascending=False)
print(importances2)

print("\nScore ranges by grade:")
print(agg.groupby('GRADE')['SCORE'].describe()[['min', 'mean', 'max']])

SCORE                  0.656142
CRITICAL_VIOLATIONS    0.233999
TOTAL_VIOLATIONS       0.096773
CUISINE                0.010509
BORO                   0.002576
dtype: float64

Score ranges by grade:
       min       mean   max
GRADE                      
A      0.0   8.999619  40.0
B      0.0  19.747447  43.0
C      0.0  30.362708  86.0


In [26]:
features3 = ['BORO', 'CUISINE', 'TOTAL_VIOLATIONS', 'CRITICAL_VIOLATIONS']

X3_train = X2_train[features3]
X3_test = X2_test[features3]

model4 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42,
                                 n_jobs=-1, class_weight='balanced')
model4.fit(X3_train, y2_train)

y_pred4 = model4.predict(X3_test)
y_proba4 = model4.predict_proba(X3_test)[:, 1]

print(classification_report(y2_test, y_pred4, target_names=['Not A', 'A']))
print(f'ROC-AUC: {roc_auc_score(y2_test, y_proba4):.4f}')

importances3 = pd.Series(model4.feature_importances_, index=features3).sort_values(ascending=False)
print(f'\nFeature importances:\n{importances3}')

              precision    recall  f1-score   support

       Not A       0.55      0.84      0.66      1861
           A       0.98      0.91      0.94     14686

    accuracy                           0.90     16547
   macro avg       0.76      0.88      0.80     16547
weighted avg       0.93      0.90      0.91     16547

ROC-AUC: 0.9445

Feature importances:
CRITICAL_VIOLATIONS    0.633513
TOTAL_VIOLATIONS       0.317407
CUISINE                0.038755
BORO                   0.010325
dtype: float64


In [27]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y2_test, y_pred4)
tn, fp, fn, tp = cm.ravel()

print("=== Final Model Summary ===")
print(f"Task       : Predict Grade A vs Not A")
print(f"Features   : {features3}")
print(f"Train size : {X3_train.shape[0]:,}  |  Test size: {X3_test.shape[0]:,}")
print(f"\nROC-AUC    : {roc_auc_score(y2_test, y_proba4):.4f}")
print(f"Accuracy   : {(y_pred4 == y2_test).mean():.4f}")
print(f"\nConfusion Matrix:")
print(f"  True Not-A  : {tn}  |  False A    : {fp}")
print(f"  False Not-A : {fn}  |  True A     : {tp}")

=== Final Model Summary ===
Task       : Predict Grade A vs Not A
Features   : ['BORO', 'CUISINE', 'TOTAL_VIOLATIONS', 'CRITICAL_VIOLATIONS']
Train size : 66,186  |  Test size: 16,547

ROC-AUC    : 0.9445
Accuracy   : 0.9043

Confusion Matrix:
  True Not-A  : 1564  |  False A    : 297
  False Not-A : 1287  |  True A     : 13399


# NYC Restaurant Inspection — ML Model Summary

## What We Built
A binary classifier that predicts whether a NYC restaurant earns an **A grade** from a health inspection, using only information available at inspection time — no score leakage.

---

## Journey Summary

| Attempt | Target | ROC-AUC | Issue |
|---|---|---|---|
| 1 | Critical Flag (w/ SCORE) | 0.63 | SCORE dominated, weak features |
| 2 | Critical Flag (no SCORE) | 0.53 | Near random, no real signal |
| 3 | Grade A (w/ SCORE) | 0.99 | SCORE = grade, pure leakage |
| **4** | **Grade A (no SCORE)** | **0.94** | ✅ Clean model |

---

## Final Model

- **Algorithm:** Random Forest (100 trees, max depth 10, balanced classes)
- **Features:** `CRITICAL_VIOLATIONS`, `TOTAL_VIOLATIONS`, `CUISINE`, `BORO`
- The model correctly catches **84% of non-A restaurants** — useful for flagging risky places
